# Surrogate Model Training (MLP)\n
\n
This notebook trains a **Neuro-Fuzzy–inspired neural surrogate model** to approximate the ANFIS reasoning layer.\n
While a classical ANFIS consists of explicit membership functions and fuzzy rules, this surrogate neural model learns a smooth mapping that emulates the same reasoning behaviour for real-time deployment in the game engine.

In [1]:
import pandas as pd
import os
import json
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

INPUT_PATH = 'data/processed/6_anfis_dataset.csv'
MODEL_OUT_PATH = 'data/models/anfis_params.json'

print("Loading dataset...")
if os.path.exists(INPUT_PATH):
    df = pd.read_csv(INPUT_PATH)
    
    # Features & Target
    X = df[['pct_combat', 'pct_collect', 'pct_explore', 'delta_combat', 'delta_collect', 'delta_explore']]
    y = df['target_multiplier']
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Initialize Model
    # Compact architecture to ensure runtime efficiency in game engine
    # 6 Inputs -> 16 Hidden -> 8 Hidden -> 1 Output
    print("Training MLP Regressor...")
    mlp = MLPRegressor(hidden_layer_sizes=(16, 8), activation='relu', solver='adam', max_iter=1000, random_state=42)
    mlp.fit(X_train, y_train)
    
    # Evaluate
    y_pred = mlp.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    print(f"\nEvaluation:\nMSE: {mse:.4f}\nMAE: {mae:.4f}")
    
    # Export Weights
    # Accessing weights: coefs_ (weights), intercepts_ (biases)
    # This saves learned model parameters to a JSON file for import into the runtime game engine inference layer.\n
    params = {
        'layer_1_weights': mlp.coefs_[0].tolist(),
        'layer_1_bias': mlp.intercepts_[0].tolist(),
        'layer_2_weights': mlp.coefs_[1].tolist(),
        'layer_2_bias': mlp.intercepts_[1].tolist(),
        'output_layer_weights': mlp.coefs_[2].tolist(),
        'output_layer_bias': mlp.intercepts_[2].tolist(),
        'activation': mlp.activation
    }
    
    os.makedirs('data/models', exist_ok=True)
    with open(MODEL_OUT_PATH, 'w') as f:
        json.dump(params, f, indent=4)
        
    print(f"\nModel parameters exported to {MODEL_OUT_PATH}")
else:
    print(f"Error: {INPUT_PATH} not found.")

ModuleNotFoundError: No module named 'sklearn'